# StyleTTS 2 Demo (CZ)

In [2]:
# Go to the root of the repository
%cd ..

/auto/plzen4-ntis/home/jmatouse/experimenty/StyleTTS2_plbert-cs-epa


/usr/local/lib/python3.10/dist-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


### Randomness

In [3]:
import torch

torch.manual_seed(0)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

import random

random.seed(0)

import numpy as np

np.random.seed(0)

### Imports and packages

In [4]:
# load packages
import sys
import os
import time
import yaml
import torch
import torch.nn.functional as F
import torchaudio
# import librosa
# from nltk.tokenize import word_tokenize

from models import load_ASR_models, load_F0_models, build_model
from Modules.diffusion.sampler import (ADPM2Sampler, DiffusionSampler,
                                       KarrasSchedule)
from Utils.PLBERT.util import load_plbert
from utils import recursive_munch
from text_utils2 import TextCleaner

from tpp_ttstool import TppTtstool

%matplotlib inline
import IPython.display as ipd

### Functions and definitions

In [5]:
to_mel = torchaudio.transforms.MelSpectrogram(
    n_mels=80, n_fft=2048, win_length=1200, hop_length=300
)
mean, std = -4, 4


def length_to_mask(lengths):
    mask = torch.arange(lengths.max()).unsqueeze(0).expand(lengths.shape[0], -1).type_as(lengths)
    mask = torch.gt(mask + 1, lengths.unsqueeze(1))
    return mask


def preprocess(wave):
    wave_tensor = torch.from_numpy(wave).float()
    mel_tensor = to_mel(wave_tensor)
    mel_tensor = (torch.log(1e-5 + mel_tensor.unsqueeze(0)) - mean) / std
    return mel_tensor


# def compute_style(ref_dicts, model):
#     reference_embeddings = {}
#     for key, path in ref_dicts.items():
#         wave, sr = librosa.load(path, sr=24000)
#         audio, _ = librosa.effects.trim(wave, top_db=30)
#         if sr != 24000:
#             audio = librosa.resample(audio, sr, 24000)
#         mel_tensor = preprocess(audio).to(DEVICE)

#         with torch.no_grad():
#             ref = model.style_encoder(mel_tensor.unsqueeze(1))
#         reference_embeddings[key] = (ref.squeeze(1), audio)

#     return reference_embeddings

In [6]:
# Set username
USER = os.environ["USER"]

TPP_PATH = f"/storage/plzen4-ntis/home/{USER}/GIT_repos/TPP/src"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Set path to TPP
sys.path.insert(0, TPP_PATH)

# Define bin and data for phonemizer
TTSTOOL_BIN_PATH = "./Utils/tts_tool/tts_tool"
TTSTOOL_DATA_PATH = "./Utils/tts_tool/data/frontend_ph-redu.json"

### Load models

In [7]:
# Set up TPP
tpp = TppTtstool("cs-cz", tts_tool_bin=TTSTOOL_BIN_PATH, tts_tool_data=TTSTOOL_DATA_PATH)

# Load processed training config
config = yaml.safe_load(open("Exps/KleIl/config2.processed.yml"))

text_cleaner = TextCleaner(
    config["data_params"]["symbol_dict_path"], pad=config["data_params"]["pad"]
)
print(f"Number of symbols: {len(text_cleaner)}")
assert len(text_cleaner) == 81, f"Number of symbols must be 81 but it is {len(text_cleaner)}"

# Load pretrained models
text_aligner = load_ASR_models(config["ASR_path"], config["ASR_config"])  # Text aligner
pitch_extractor = load_F0_models(config["F0_path"])  # F0 extractor
plbert = load_plbert(config["PLBERT_dir"])  # PLBERT

Number of symbols: 81
Loading PL-BERT at Utils/PLBERT//step_8000000.t7 ...


In [8]:
# Build StyleTTS2 model
model = build_model(recursive_munch(config["model_params"]), text_aligner, pitch_extractor, plbert)
_ = [model[key].eval() for key in model]  # Set model to eval mode
_ = [model[key].to(DEVICE) for key in model]  # Move model to device

/usr/local/lib/python3.10/dist-packages/torch/nn/utils/weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")
/usr/local/lib/python3.10/dist-packages/torch/nn/modules/rnn.py:82: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


In [9]:
params_whole = torch.load("Exps/KleIl/second_stage.pth", map_location="cpu")
params = params_whole["net"]

In [10]:
# Hack to cope with module prefix
for key in model:
    if key in params:
        print("%s loaded" % key)
        try:
            model[key].load_state_dict(params[key])
        except:
            from collections import OrderedDict

            state_dict = params[key]
            new_state_dict = OrderedDict()
            for k, v in state_dict.items():
                name = k[7:]  # remove `module.`
                new_state_dict[name] = v
            # load params
            model[key].load_state_dict(new_state_dict, strict=False)

_ = [model[key].eval() for key in model]  # Set model to eval mode

bert loaded
bert_encoder loaded
predictor loaded
decoder loaded
text_encoder loaded
predictor_encoder loaded
style_encoder loaded
diffusion loaded
text_aligner loaded
pitch_extractor loaded
mpd loaded
msd loaded
wd loaded


In [11]:
sampler = DiffusionSampler(
    model.diffusion.diffusion,
    sampler=ADPM2Sampler(),
    sigma_schedule=KarrasSchedule(sigma_min=0.0001, sigma_max=3.0, rho=9.0),  # empirical parameters
    clamp=False,
)

### Synthesize speech

In [32]:
# synthesize a text
text = """Šestašedesátiletý nadšený hráč stolního tenisu vlastní mnoho objektů v kraji. Je to prostě borec."""

In [23]:
def synthesize(
    text,
    model,
    tpp,
    text_cleaner,
    sampler,
    diffusion_steps=5,
    embedding_scale=1,
    alpha=0.7,
    device="cuda",
):
    # Clean text
    text = text.strip()
    text = text.replace('"', "")

    # Prepare phonemizer
    tpp.ssml_parse(text)

    # Initialize previous style and wavs
    wavs = []
    s_prev = None  # reset style of previous sentence `s_prev`

    # Iterate over sentences
    for ps in tpp.to_sentences_phon():
        if not ps.strip():  # skip empty phonetic string
            continue

        tokens = [0] + text_cleaner(ps)  # add padding and tokenize phonetic string

        wav, s_prev = LFinference(
            torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0),
            model,
            sampler,
            diffusion_steps=diffusion_steps,
            embedding_scale=embedding_scale,
            s_prev=s_prev,
            alpha=alpha,
        )
        wavs.append(wav)

    return wavs

    # # Clean text
    # text = text.strip()
    # text = text.replace('"', "")

    # # Prepare phonemizer
    # tpp.ssml_parse(text)
    # wavs = []

    # for ps in tpp.to_sentences_phon():
    #     tokens = [0] + text_cleaner(ps) + [0]
    #     tokens = torch.tensor(tokens, dtype=torch.long, device=noise.device).unsqueeze(0)
    #     wav = inference(tokens, model, noise, sampler, diffusion_steps, embedding_scale)
    #     wavs.append(wav)

    # return wavs

In [25]:
def inference(tokens, model, noise, sampler, diffusion_steps=5, embedding_scale=1):
    with torch.no_grad():
        input_lengths = torch.tensor([tokens.shape[-1]], device=noise.device)
        text_mask = length_to_mask(input_lengths).to(noise.device)

        t_en = model.text_encoder(tokens, input_lengths, text_mask)
        bert_dur = model.bert(tokens, attention_mask=(~text_mask).int())
        d_en = model.bert_encoder(bert_dur).transpose(-1, -2)

        s_pred = sampler(
            noise,
            embedding=bert_dur[0].unsqueeze(0),
            num_steps=diffusion_steps,
            embedding_scale=embedding_scale,
        ).squeeze(0)

        s = s_pred[:, 128:]
        ref = s_pred[:, :128]

        d = model.predictor.text_encoder(d_en, s, input_lengths, text_mask)

        x, _ = model.predictor.lstm(d)
        duration = model.predictor.duration_proj(x)
        duration = torch.sigmoid(duration).sum(axis=-1)
        pred_dur = torch.round(duration.squeeze()).clamp(min=1)

        pred_dur[-1] += 5

        pred_aln_trg = torch.zeros(input_lengths, int(pred_dur.sum().data))
        c_frame = 0
        for i in range(pred_aln_trg.size(0)):
            pred_aln_trg[i, c_frame : c_frame + int(pred_dur[i].data)] = 1
            c_frame += int(pred_dur[i].data)

        # encode prosody
        en = d.transpose(-1, -2) @ pred_aln_trg.unsqueeze(0).to(noise.device)
        F0_pred, N_pred = model.predictor.F0Ntrain(en, s)
        out = model.decoder(
            t_en @ pred_aln_trg.unsqueeze(0).to(noise.device),
            F0_pred,
            N_pred,
            ref.squeeze().unsqueeze(0),
        )

    return out.squeeze().cpu().numpy()

In [ ]:
def LFinference(
    tokens, model, sampler, diffusion_steps=5, embedding_scale=1, s_prev=None, alpha=0.7
):
    with torch.no_grad():
        input_lengths = torch.LongTensor([tokens.shape[-1]], device=tokens.device)
        text_mask = length_to_mask(input_lengths).to(tokens.device)

        t_en = model.text_encoder(tokens, input_lengths, text_mask)
        bert_dur = model.bert(tokens, attention_mask=(~text_mask).int())
        d_en = model.bert_encoder(bert_dur).transpose(-1, -2)

        s_curr = sampler(
            torch.randn(1, 1, 256, device=DEVICE),  # noise
            embedding=bert_dur[0].unsqueeze(0),
            num_steps=diffusion_steps,
            embedding_scale=embedding_scale,
        ).squeeze(0)

        # Combine styles
        if s_prev is not None:
            s_curr = (
                alpha * s_curr + (1 - alpha) * s_prev
            )  # convex combination of previous and current styles

        s = s_curr[:, 128:]
        ref = s_curr[:, :128]

        d = model.predictor.text_encoder(d_en, s, input_lengths, text_mask)

        x, _ = model.predictor.lstm(d)
        duration = model.predictor.duration_proj(x)
        duration = torch.sigmoid(duration).sum(axis=-1)
        pred_dur = torch.round(duration.squeeze()).clamp(min=1)

        pred_aln_trg = torch.zeros(input_lengths, int(pred_dur.sum().data))
        c_frame = 0
        for i in range(pred_aln_trg.size(0)):
            pred_aln_trg[i, c_frame : c_frame + int(pred_dur[i].data)] = 1
            c_frame += int(pred_dur[i].data)

        # Encode prosody
        en = d.transpose(-1, -2) @ pred_aln_trg.unsqueeze(0).to(tokens.device)
        F0_pred, N_pred = model.predictor.F0Ntrain(en, s)
        out = model.decoder(
            t_en @ pred_aln_trg.unsqueeze(0).to(tokens.device),
            F0_pred,
            N_pred,
            ref.squeeze().unsqueeze(0),
        )

    return out.squeeze().cpu().numpy(), s_curr

In [12]:
def inference2(
    model, tpp, text, text_cleaner, noise, sampler, diffusion_steps=5, embedding_scale=1
):
    text = text.strip()
    text = text.replace('"', "")

    tpp.ssml_parse(text)
    # Only the 1st sentence is used
    ps = list(tpp.to_sentences_phon())
    print(ps[0])
    ps = word_tokenize(ps[0])
    ps = " ".join(ps)

    tokens = [0] + text_cleaner(ps) + [0]
    # tokens = torch.LongTensor(tokens, device=DEVICE).unsqueeze(0)
    tokens = torch.tensor(tokens, dtype=torch.long, device=DEVICE).unsqueeze(0)

    with torch.no_grad():
        # input_lengths = torch.LongTensor([tokens.shape[-1]], device=tokens.device)
        input_lengths = torch.tensor([tokens.shape[-1]], device=tokens.device)
        text_mask = length_to_mask(input_lengths).to(tokens.device)

        t_en = model.text_encoder(tokens, input_lengths, text_mask)
        bert_dur = model.bert(tokens, attention_mask=(~text_mask).int())
        d_en = model.bert_encoder(bert_dur).transpose(-1, -2)

        s_pred = sampler(
            noise,
            embedding=bert_dur[0].unsqueeze(0),
            num_steps=diffusion_steps,
            embedding_scale=embedding_scale,
        ).squeeze(0)

        s = s_pred[:, 128:]
        ref = s_pred[:, :128]

        d = model.predictor.text_encoder(d_en, s, input_lengths, text_mask)

        x, _ = model.predictor.lstm(d)
        duration = model.predictor.duration_proj(x)
        duration = torch.sigmoid(duration).sum(axis=-1)
        pred_dur = torch.round(duration.squeeze()).clamp(min=1)

        pred_dur[-1] += 5

        pred_aln_trg = torch.zeros(input_lengths, int(pred_dur.sum().data))
        c_frame = 0
        for i in range(pred_aln_trg.size(0)):
            pred_aln_trg[i, c_frame : c_frame + int(pred_dur[i].data)] = 1
            c_frame += int(pred_dur[i].data)

        # encode prosody
        en = d.transpose(-1, -2) @ pred_aln_trg.unsqueeze(0).to(DEVICE)
        F0_pred, N_pred = model.predictor.F0Ntrain(en, s)
        out = model.decoder(
            t_en @ pred_aln_trg.unsqueeze(0).to(DEVICE), F0_pred, N_pred, ref.squeeze().unsqueeze(0)
        )

    return out.squeeze().cpu().numpy()

In [13]:
# import nltk

# nltk.download("punkt")

[nltk_data] Downloading package punkt to
[nltk_data]     /storage/plzen4-ntis/home/jmatouse/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

#### Basic synthesis (5 diffusion steps)

In [33]:
start = time.time()
# noise = torch.randn(1, 1, 256, device=DEVICE)
# wavs = synthesize(text, model, tpp, text_cleaner, noise, sampler, diffusion_steps=5, embedding_scale=1)
wavs = synthesize(text, model, tpp, text_cleaner, sampler, diffusion_steps=5, embedding_scale=1)
# wav = inference(model, tpp, text, text_cleaner, noise, sampler, diffusion_steps=5, embedding_scale=1)
wav = np.concatenate(wavs)
rtf = (time.time() - start) / (len(wav) / 24000)
print(f"RTF = {rtf:5f}")

display(ipd.Audio(wav, rate=24000))

RTF = 0.121209


#### With higher diffusion steps (more diverse)
Since the sampler is ancestral, the higher the steps, the more diverse the samples are, with the cost of slower synthesis speed.

In [34]:
start = time.time()
noise = torch.randn(1, 1, 256, device=DEVICE)
wav = inference(
    model, tpp, text, text_cleaner, noise, sampler, diffusion_steps=10, embedding_scale=1
)
rtf = (time.time() - start) / (len(wav) / 24000)
print(f"RTF = {rtf:5f}")
import IPython.display as ipd

display(ipd.Audio(wav, rate=24000))

SestaSedesATiletI natSenI hrAC stolJIho tenisu vlastJI mnoho objektU f kraji.
RTF = 0.142207


### Speech expressiveness
The following section recreates the samples shown in [Section 6](https://styletts2.github.io/#emo) of the demo page.

#### With embedding_scale=1
This is the classifier-free guidance scale. The higher the scale, the more conditional the style is to the input text and hence more emotional. 

In [21]:
texts = {}
texts["Happy"] = "Já jsem tak šťastný, že tomu ani nemohu uvěřit."
texts["Sad"] = "Je mi to strašně líto, ale já jsem s tím nemohl nic dělat."
texts["Angry"] = "Tak to jdi do prdele! Tohle se nedělá"
texts["Surprised"] = "Tak tomu nemohu uvěřit! Opravdu se to stalo?"

for k, v in texts.items():
    noise = torch.randn(1, 1, 256, device=DEVICE)
    wav = inference(
        model, tpp, v, text_cleaner, noise, sampler, diffusion_steps=10, embedding_scale=1
    )
    print(k + ": ")
    display(ipd.Audio(wav, rate=24000, normalize=True))

jA sem tak STastnI, Ze tomu aJi nemohu uvjeRit.
Happy: 


je mi to straSJe lIto, ale jA sem s TIm nemohl Jidz Delat.
Sad: 


tak to jDi do prdele!
Angry: 


tak tomu nemohu uvjeRit!
Surprised: 


#### With embedding_scale=2

In [27]:
texts = {}
texts["Happy"] = "Já jsem tak šťastný, že tomu ani nemohu uvěřit."
texts["Sad"] = "Je mi to strašně líto, ale já jsem s tím nemohl nic dělat."
texts["Angry"] = "Tak to jdi do prdele! Tohle se nedělá"
texts["Surprised"] = "Tak tomu nemohu uvěřit! Opravdu se to stalo?"

for k, v in texts.items():
    noise = torch.randn(1, 1, 256, device=DEVICE)
    wav = inference(
        model, tpp, v, text_cleaner, noise, sampler, diffusion_steps=10, embedding_scale=4
    )  # embedding_scale=2 for more pronounced emotion
    print(k + ": ")
    display(ipd.Audio(wav, rate=24000, normalize=True))

jA sem tak STastnI, Ze tomu aJi nemohu uvjeRit.
Happy: 


je mi to straSJe lIto, ale jA sem s TIm nemohl Jidz Delat.
Sad: 


tak to jDi do prdele!
Angry: 


tak tomu nemohu uvjeRit!
Surprised: 


### Long-form generation
This section includes basic implementation of Algorithm 1 in the paper for consistent longform audio generation. The example passage is taken from [Section 5](https://styletts2.github.io/#long) of the demo page. 

In [ ]:
passage = """Šestašedesátiletý nadšený hráč stolního tenisu vlastní a spravuje většinu areálu mezi Budvarem a Ternem v krajském městě. Zbytek má jeho manželka. V rozhovoru vzpomíná na první roky v nové zemi, na boom zmíněných stánků u hranic i to, proč si vietnamská menšina u nás získala své místo."""

In [ ]:
def LFinference2(
    text, text_cleaner, s_prev, noise, alpha=0.7, diffusion_steps=5, embedding_scale=1
):
    text = text.strip()
    text = text.replace('"', "")
    ps = global_phonemizer.phonemize([text])  # TODO: tts_tool
    ps = word_tokenize(ps[0])
    ps = " ".join(ps)

    tokens = [0] + text_cleaner(ps) + [0]
    tokens = torch.LongTensor(tokens, device=DEVICE).unsqueeze(0)

    with torch.no_grad():
        input_lengths = torch.LongTensor([tokens.shape[-1]], device=tokens.device)
        text_mask = length_to_mask(input_lengths).to(tokens.device)

        t_en = model.text_encoder(tokens, input_lengths, text_mask)
        bert_dur = model.bert(tokens, attention_mask=(~text_mask).int())
        d_en = model.bert_encoder(bert_dur).transpose(-1, -2)

        s_pred = sampler(
            noise,
            embedding=bert_dur[0].unsqueeze(0),
            num_steps=diffusion_steps,
            embedding_scale=embedding_scale,
        ).squeeze(0)

        if s_prev is not None:
            # convex combination of previous and current style
            s_pred = alpha * s_prev + (1 - alpha) * s_pred

        s = s_pred[:, 128:]
        ref = s_pred[:, :128]

        d = model.predictor.text_encoder(d_en, s, input_lengths, text_mask)

        x, _ = model.predictor.lstm(d)
        duration = model.predictor.duration_proj(x)
        duration = torch.sigmoid(duration).sum(axis=-1)
        pred_dur = torch.round(duration.squeeze()).clamp(min=1)

        pred_aln_trg = torch.zeros(input_lengths, int(pred_dur.sum().data))
        c_frame = 0
        for i in range(pred_aln_trg.size(0)):
            pred_aln_trg[i, c_frame : c_frame + int(pred_dur[i].data)] = 1
            c_frame += int(pred_dur[i].data)

        # encode prosody
        en = d.transpose(-1, -2) @ pred_aln_trg.unsqueeze(0).to(DEVICE)
        F0_pred, N_pred = model.predictor.F0Ntrain(en, s)
        out = model.decoder(
            t_en @ pred_aln_trg.unsqueeze(0).to(DEVICE), F0_pred, N_pred, ref.squeeze().unsqueeze(0)
        )

    return out.squeeze().cpu().numpy(), s_pred

In [ ]:
sentences = passage.split(".")  # simple split by comma
wavs = []
s_prev = None
for text in sentences:
    if text.strip() == "":
        continue
    text += "."  # add it back
    noise = torch.randn(1, 1, 256, device=DEVICE)
    wav, s_prev = LFinference(
        text, s_prev, noise, alpha=0.7, diffusion_steps=10, embedding_scale=1.5
    )
    wavs.append(wav)
display(ipd.Audio(np.concatenate(wavs), rate=24000, normalize=False))